In [ ]:
# This notebook requires GPU

# read in the dataset 

# replace the argument below with the filepath that we have copied 
# we use the open() function to open the file, returning a file object
# subsequently, we use the read() method to read from the file, saving the info into "data"
data = open('/kaggle/input/english-poems-dataset/poems.txt').read()

In [ ]:
# If there is an error later in importing keras-nlp
# 1. stop the session
# 2. comment out this line, and rerun
# !pip install tf_keras

In [ ]:
# We use the approach of extending our dataset that we discussed in our slides
# import the necessary libraries

import tensorflow as tf
from tensorflow import keras
import numpy as np

In [ ]:
# instead of splitting by newline, we will generate more data by creating a window 
# of size 10 words, and progressively moving it one word at a time

window_size = 10
alltext = []

lowercase = data.lower()
# remove newline characters, since we are not splitting by newline
nonewline = lowercase.replace('\n', ' ')

# remove punctuation
import string
nopunct = nonewline.translate(str.maketrans('', '', string.punctuation))

# transfer the entire text of poems into a list
alltext.append(nopunct)
# split into individual words and store in 'words'
words = nopunct.split(" ")

# this is the total number of words across all the poems in the dataset
len(words)

In [ ]:
# we see that punctuation and newlines have been replaced (reference only)
alltext

In [ ]:
# we see that the last word is actually a blank, since we cut words using " " and we ended with a space
# so we trim out this last "word"
words[14142]

In [ ]:
# we clear alltext
alltext = []

# we subset by removing the last character
nopunct = nopunct[:-1]
alltext.append(nopunct)
words = nopunct.split(" ")
len(words)

In [ ]:
# last word is now okay
words[14141]

In [ ]:
# we also notice between some words, there are multiple spaces, as between the second last and
# last words here
# 'i have wished a bird would fly away and not sing by my house all day  have...'
# this will result in some blank words in our list too, as shown
words[:20]

In [ ]:
# we include only the actual words in our list
new_words = []

for word in words: 
    if word != '':
        new_words.append(word)

words = new_words

# checking
words[:20]

In [ ]:
len(words)

In [ ]:
# create an empty list called "sentences"
sentences = []

In [ ]:
# DO NOT RUN THIS MULTIPLE TIMES, BECAUSE IT WILL KEEP ADDING SEQUENCES TO SENTENCES
# THE CORRECT OUTPUT YOU SHOULD GET HERE IS 13832
# If you don't get the answer, you can run the above chunk to reset sentences to []

range_size = len(words) - (window_size - 1)
for i in range(0, range_size):
    current_sentence = ""
    
    # for word in range(0, window_size - 1):
    for word in range(0, window_size):
        word = words[i + word]
        current_sentence = current_sentence + word
        current_sentence = current_sentence + " "
    sentences.append(current_sentence)

len(sentences)

# we should get (number of words) - (window size - 1) number of sentences
# E.g. if we have a sentence of 4 words with a window of 3 words, we would get 2 sentences
# Is this the expected number of sentences we get? Let's go to the slides

In [ ]:
# first sentence
sentences[0]

In [ ]:
# second sentence
sentences[1]

In [ ]:
# last sentence
sentences[13831]

#### Demo for the above code:

Let's say your sentence is: I am a Marvel superhero and my powers are flight 

And we chose a window of 6 words

range_size = len(words) - (window_size - 1) = 10 - 5 = 5

So we repeat for i = 0, 1, 2, 3, 4

the first time around, i = 0, and we loop through words from 0 to window_size, i.e. 6 (which means the first six words)

word = words[0 + 0] = "I ", and the current sentence is "I"

The next time around, word = words[0 + 1] = "am", and current_sentence becomes "I am "

And at the end of repeating this six times, we get the first sentence "I am a Marvel superhero and " appended to the list "sentences"

So the next time around, for i = 1, we extract six words starting with the second word now
Since word = words[i + word] = words[1 + 0] = words[1]
Thus, we extract the sentence "am a Marvel superhero and my "

And finally for i = 4, we extract the words with index 4 till 9, or "superhero and my powers are flight"

#### The next few chunks are similar to the previous notebook, so we can just run them and skip to where we set the max_sequence_length to the window size

In [ ]:
# we will use the Tokenizer for text vectorization, as what we learned in our earlier class
from tensorflow.keras.preprocessing.text import Tokenizer

# specify the OOV token; the OOV token will then be added to word_index and used to replace 
# out-of-vocabulary words during text_to_sequence calls 
oov_tok = ""

# instantiate our Tokenizer
tokenizer = Tokenizer(oov_token=oov_tok, split=" ", char_level = False)

# learn the vocab from all of the poems
tokenizer.fit_on_texts(alltext)

# set the number of words in our vocabulary to the number in our word index + 1 
total_words = len(tokenizer.word_index) + 1

# check how many words we have in our vocab
print(total_words)

In [ ]:
# formally set the vocab size to the total number of words
vocab_size = total_words

In [ ]:
# why do we need to add 1 to the length of tokenizer.word_index to get the vocab_size? 
# our tokenizer.word_index is a Python dictionary of key:value pairs, starting with 
# key = 1 for the first token, the OOV
print(tokenizer.word_index)

# we see below that the last key value pair is 3080: 'nay', meaning we have 3080 words in our vocab
# but for Python indexing, the last index is exclusive, so we use 3080 as the upper index, we won't be 
# able to index this last word; hence we need a vocab size of 3080 + 1, to be able to index up to the last word

In [ ]:
# this is the first line, containing 6 words
sentences[0]

In [ ]:
# vectorize each line

input_sequences = tokenizer.texts_to_sequences(sentences)

# take a look at the first 20 members of our list
input_sequences[:20]
# indeed we see that we tokenize each line

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

# we set the max_sequence_length to the window size, and convert to an array
max_sequence_length = window_size

input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_length, padding='pre'))
input_sequences[:5]

In [ ]:
# create features and label

# each feature would be the vectorized sentence up till and excluding the last word
# the label would be the vectorized sentence from the second word onwards

features, labels = input_sequences[:,:-1], input_sequences[:,1:]

In [ ]:
# this is the first vectorized sentence
input_sequences[0]

In [ ]:
# this is the first feature
features[0]

In [ ]:
# this is the first label
labels[0]

In [ ]:
# check the shape for our features array
features.shape

In [ ]:
# check the shape for our labels array; let's go back to our notebooks to discuss 
labels.shape

#### The next few chunks on assembling the NN are similar to the previous notebook
#### So we can jump ahead to where we plot the graphs

In [ ]:
# install the library that we need for transformers

# need keras 3 for this notebook 
# !pip install keras-nlp --upgrade
!pip install keras-nlp 

In [ ]:
# do the necessary imports
import keras_nlp
from keras import initializers
from tensorflow.keras import layers

In [ ]:
# we build the model, using the same parameters as our last class

embed_dim = 64
layer_dim = 16
num_heads = 2

# we instantiate the decoder; 
# the TransformerDecoder class will always apply causal masking to the decoder attention layer
# so we don't have to explicitly set it using an argument, unlike the padding mask 
decoder = keras_nlp.layers.TransformerDecoder(intermediate_dim = layer_dim, num_heads = num_heads, 
                                             kernel_initializer = initializers.glorot_uniform(seed = 0))

# our inputs are 61-dimensional padded feature vectors
inputs = keras.Input(shape=((max_sequence_length - 1),), dtype="int64")
# for each word in our vocab, we generate a 64-dimensional embedding vector
x = layers.Embedding(vocab_size, embed_dim, mask_zero = True)(inputs)

# we add the positional encoding vectors to our word vectors, to get new position-aware embedding vectors
# to do this, we generate the position encoding layer first using the SinePositionEncoding class
position_encoding = keras_nlp.layers.SinePositionEncoding()(x)
# then we add the position encoding layer to our embedding layer 
new_position_encoding = x + position_encoding

# we pass the position-aware embedding vectors through the decoder
x = decoder(new_position_encoding)

# code for later; comment out first
# x = decoder(x)

# Let's go to our slides to talk more on the output layer
outputs = layers.Dense(vocab_size, activation="softmax")(x)


model = keras.Model(inputs, outputs)

# set our model for training
# similar to our choice of activation function above, "sparse categorical crossentropy" is just 
# an extension of "binary crossentropy", but for multi-class classification
model.compile(loss="sparse_categorical_crossentropy", optimizer="rmsprop", metrics = ['accuracy'])

# we train for 150 epochs
num_epochs = 150

# we save the training history into a variable so we can plot the results subsequently
history = model.fit(features, labels, epochs = num_epochs, verbose = 1)

In [ ]:
import matplotlib.pyplot as plt

def plot_graphs(process, metric, filename):      
    plt.plot(process.history[metric])
    plt.title(filename.split('.')[0])
    plt.xlabel("Epochs")
    plt.ylabel(metric)
    plt.savefig(filename)
    plt.show()

In [ ]:
# plotting the results
plot_graphs(history, "accuracy", "Poems - Transformer Accuracy.png")
plot_graphs(history, "loss", "Poems - Transformer Loss.png")

# What do we think about the results? Let's go to the slides to discuss

In [ ]:
# let's test it on an existing sentence (that is part of the training set)
# we will see if it is able to predict "love", the last word based on the preceding 
# sentence "Or like the poetess's heart of"
seed_text = "Or like the poetess's heart of"

# we tokenize this sentence, and extracting the one list of tokens that we have by indexing [0]
token_list = tokenizer.texts_to_sequences([seed_text])[0]
token_list

In [ ]:
# we now need to pad this to get it into the same shape as the data used for training
# since our features were one token less than the max sequence length, we have use max_sequence_length - 1
token_list = pad_sequences([token_list], maxlen = max_sequence_length - 1, padding = 'pre')
token_list

In [ ]:
# now we predict the next word by calling model.predict()
# this will return the probabilities for each word in the corpus
# we get the one with the highest probability, using the argmax() function from numpy

# axis = -1 means the last dimension of the result returned by the argmax() function, 
# which contains the indexed token with the highest probability for that particular position 
# in the sequence
predicted = np.argmax(model.predict(token_list), axis = -1)
print(predicted)

# we get the predicted vectorized sequence

In [ ]:
# the last word really is what we are interested in
# we got a numpy array as output so we need to index twice to get the last token
predicted_word = predicted[-1][-1]
predicted_word

In [ ]:
# The previous chunk gave us the predicted word index; let's look up what this word is
for word, i in tokenizer.word_index.items():
    if i == predicted_word: 
        print(word)
        break

### Is the result correct? 

In [ ]:
# Let's start generating text from our model!
# we begin with a seed text; you can try a different seed text if you like, and contrast the results

start_text = "The flowers bloom"
# we will predict the next 100 words
next_words = 100

# repeat these steps for each word, for the next 100 words we are trying to predict
for _ in range(next_words):
    # we vectorize the current text at this point, and pad it
    token_list = tokenizer.texts_to_sequences([start_text])[0]
    token_list = pad_sequences([token_list], maxlen = max_sequence_length - 1, padding = 'pre')
    # predict the token for the next word, using predict(); this gives a list of 9 tokens
    # which are the most likely at their respective locations, as retrieved by argmax()
    predicted = np.argmax(model.predict(token_list), axis = -1)
    # extract the last token which corresponds to the next word we are predicting
    predicted_word = predicted[-1][-1]
    
    # create an empty string to store the predicted word
    output_word = ''
    
    # check what the predicted word is, based on the tokenized index, and store in output_word
    for word, i in tokenizer.word_index.items():
        if i == predicted_word: 
            output_word = word
            break
    
    # append the predicted word to the seed text
    start_text += " " + output_word

# let's see our result 
print(start_text)

### Note that punctuation wasn't part of our vocabulary, so our generated text won't have punctuation either. What predicted text did we get? Does it seem "poem like"? Let's go to our slides to discuss